In [1]:
import os
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import zoom
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAJ_DIR   = "Phase3_Data/trajectories/"
IMG_DIR    = "Phase3_Data/generated/experimental/"   # 7000 experimental images
PHASE4_JSON = "Phase4_Results/phase4_results.json"

# Output folders for Phase 7
os.makedirs("Phase7_Results/heatmaps",          exist_ok=True)  # raw 64×64 .npy heatmaps
os.makedirs("Phase7_Results/overlays",          exist_ok=True)  # 512×512 overlay PNGs
os.makedirs("Phase7_Results/gallery",           exist_ok=True)  # qualitative figures

print("All paths ready.")

All paths ready.


In [2]:
with open(PHASE4_JSON) as f:
    records = json.load(f)

print(f"Total records: {len(records)}")
print(f"Example key:   {records[0]['key']}")

# Verify: does key + .npy exist in trajectories folder?
sample_key  = records[0]['key']
sample_file = os.path.join(TRAJ_DIR, sample_key + ".npy")
exists      = os.path.exists(sample_file)
print(f"Trajectory file exists for sample key: {exists}")

# Quick count — how many of the 7000 keys have a matching trajectory file?
missing = [r['key'] for r in records if not os.path.exists(
               os.path.join(TRAJ_DIR, r['key'] + ".npy"))]
print(f"Missing trajectory files: {len(missing)}")  # should be 0

Total records: 7000
Example key:   experimental__SC1__s000__cfg0.0
Trajectory file exists for sample key: True
Missing trajectory files: 0


In [3]:
def compute_spatial_variance(traj_path: str) -> np.ndarray:
    """
    Given a trajectory file of shape (15, 4, 64, 64),
    returns a (64, 64) spatial variance map.

    Steps:
      1. var across axis=0 (15 denoising steps) → (4, 64, 64)
         This tells us: for each channel and each spatial pixel,
         how much did the x0 prediction fluctuate over the window?
      2. mean across axis=0 (4 latent channels) → (64, 64)
         Average over channels gives one scalar per spatial location.
    """
    traj = np.load(traj_path).astype(np.float32)   # (15, 4, 64, 64)
    step_var    = traj.var(axis=0)                  # (4, 64, 64)
    spatial_map = step_var.mean(axis=0)             # (64, 64)
    return spatial_map


def upsample_heatmap(heatmap_64: np.ndarray, target: int = 512) -> np.ndarray:
    """
    Bilinear upsample a (64, 64) map to (target, target).
    zoom factor = 512/64 = 8
    """
    factor = target / heatmap_64.shape[0]
    return zoom(heatmap_64, factor, order=1)   # order=1 = bilinear


def make_overlay(image_pil: Image.Image,
                 heatmap_512: np.ndarray,
                 alpha: float = 0.55) -> Image.Image:
    """
    Blend original image with a colourmap heatmap overlay.
    Returns a PIL Image.
    """
    # Normalise heatmap to [0, 1]
    h_min, h_max = heatmap_512.min(), heatmap_512.max()
    if h_max > h_min:
        heatmap_norm = (heatmap_512 - h_min) / (h_max - h_min)
    else:
        heatmap_norm = np.zeros_like(heatmap_512)

    # Apply 'inferno' colormap (black→red→yellow = low→high variance)
    colormap   = cm.get_cmap('inferno')
    heatmap_rgb = (colormap(heatmap_norm)[:, :, :3] * 255).astype(np.uint8)
    heatmap_img = Image.fromarray(heatmap_rgb).resize(
                      (512, 512), Image.BILINEAR)

    # Blend
    base    = image_pil.convert("RGB").resize((512, 512))
    blended = Image.blend(base, heatmap_img, alpha=alpha)
    return blended


# Quick sanity check on one record
r0   = records[0]
sm   = compute_spatial_variance(os.path.join(TRAJ_DIR, r0['key'] + ".npy"))
sm_up = upsample_heatmap(sm)

print(f"64×64 map  — min: {sm.min():.6f}  max: {sm.max():.6f}  mean: {sm.mean():.6f}")
print(f"512×512 map — shape: {sm_up.shape}")

# Sanity cross-check: mean of spatial map should be close to 
# the global variance already stored in phase4_results
# (They won't be identical because global var was computed differently,
#  but they should be in the same order of magnitude)
print(f"\nSpatial map mean:    {sm.mean():.6f}")
print(f"Stored global var:   {r0['variance']:.6f}")
print(f"Ratio (should be ~1-3x): {sm.mean()/r0['variance']:.2f}")

64×64 map  — min: 0.000109  max: 0.027079  mean: 0.003341
512×512 map — shape: (512, 512)

Spatial map mean:    0.003341
Stored global var:   0.003341
Ratio (should be ~1-3x): 1.00


In [4]:
# Check what the actual image filenames look like
img_files = os.listdir("Phase3_Data/generated/experimental/")
print(f"Total images: {len(img_files)}")
print(f"Example 1: {img_files[0]}")
print(f"Example 2: {img_files[1]}")

# Test the mapping
test_key = records[0]['key']  # "experimental__SC1__s000__cfg0.0"
stripped  = test_key.replace("experimental__", "")
expected_path = f"Phase3_Data/generated/experimental/{stripped}.png"
print(f"\nTest key: {test_key}")
print(f"Expected image path: {expected_path}")
print(f"File exists: {os.path.exists(expected_path)}")

Total images: 7000
Example 1: experimental__AM1__s000__cfg0.0.png
Example 2: experimental__AM1__s000__cfg1.0.png

Test key: experimental__SC1__s000__cfg0.0
Expected image path: Phase3_Data/generated/experimental/SC1__s000__cfg0.0.png
File exists: False


In [5]:
SAVE_OVERLAYS_FOR = {"both", "geometric"}  # only save PNGs for interesting cases

failed = []

for rec in tqdm(records, desc="Computing spatial variance maps"):
    key          = rec['key']
    traj_file    = os.path.join(TRAJ_DIR, key + ".npy")
    failure_type = rec['failure_type']

    # ── 1. Compute spatial variance map ──────────────────────────────────────
    try:
        spatial_map = compute_spatial_variance(traj_file)
    except Exception as e:
        failed.append((key, str(e)))
        continue

    # ── 2. Save raw 64×64 heatmap (all 7,000 images) ─────────────────────────
    np.save(os.path.join("Phase7_Results/heatmaps", key + "_heatmap.npy"),
            spatial_map)

    # ── 3. Save 512×512 overlay PNG (hallucinated + geometric only) ──────────
    if failure_type in SAVE_OVERLAYS_FOR:
        # FIXED: image filename = key + ".png" (no stripping needed)
        img_file = os.path.join(IMG_DIR, key + ".png")

        if os.path.exists(img_file):
            img     = Image.open(img_file)
            hmap_up = upsample_heatmap(spatial_map)
            overlay = make_overlay(img, hmap_up, alpha=0.55)
            overlay.save(os.path.join("Phase7_Results/overlays",
                                      key + "_overlay.png"))
        else:
            failed.append((key, "image file not found"))

print(f"\nDone. Failed: {len(failed)}")
if failed:
    print("Failures:", failed[:5])

Computing spatial variance maps:   0%|          | 0/7000 [00:00<?, ?it/s]C:\Users\Hp\AppData\Local\Temp\ipykernel_50148\2476491925.py:43: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap   = cm.get_cmap('inferno')
Computing spatial variance maps: 100%|██████████| 7000/7000 [03:31<00:00, 33.03it/s]


Done. Failed: 0


In [6]:
heatmap_files = os.listdir("Phase7_Results/heatmaps/")
overlay_files = os.listdir("Phase7_Results/overlays/")

print(f"Heatmap .npy files saved: {len(heatmap_files)}")   # should be 7000
print(f"Overlay .png files saved: {len(overlay_files)}")   # should be 574+364 = 938

# Load one heatmap and check it
sample_hmap = np.load(f"Phase7_Results/heatmaps/{heatmap_files[0]}")
print(f"\nSample heatmap shape: {sample_hmap.shape}")      # (64, 64)
print(f"min: {sample_hmap.min():.6f}  max: {sample_hmap.max():.6f}  mean: {sample_hmap.mean():.6f}")

Heatmap .npy files saved: 7000
Overlay .png files saved: 938

Sample heatmap shape: (64, 64)
min: 0.000109  max: 0.027079  mean: 0.003341


In [7]:
def spatial_concentration_index(heatmap_64: np.ndarray) -> float:
    """
    Measures how concentrated (localized) the variance is.
    
    Method: Gini coefficient of the flattened heatmap values.
    - Gini = 0.0 → perfectly uniform (variance spread everywhere equally)
    - Gini = 1.0 → perfectly concentrated (all variance in one pixel)
    
    Hallucinated images should have HIGHER Gini than clean images
    if mode interpolation is spatially localized.
    """
    flat = heatmap_64.flatten().astype(np.float64)
    flat = np.sort(flat)                          # sort ascending
    n    = len(flat)
    cumsum = np.cumsum(flat)
    # Gini formula
    gini = (2 * np.sum((np.arange(1, n+1) * flat))) / (n * cumsum[-1]) - (n+1)/n
    return float(gini)


def peak_variance_ratio(heatmap_64: np.ndarray, top_pct: float = 0.1) -> float:
    """
    Ratio of mean variance in the top X% pixels 
    vs mean variance in the bottom (1-X)% pixels.
    
    High ratio = variance is concentrated in a small hot region.
    Low ratio  = variance is spread uniformly.
    
    top_pct=0.1 means top 10% of pixels (≈410 pixels out of 4096).
    """
    flat     = heatmap_64.flatten()
    threshold = np.percentile(flat, (1 - top_pct) * 100)
    top_mean  = flat[flat >= threshold].mean()
    bot_mean  = flat[flat <  threshold].mean()
    if bot_mean == 0:
        return 0.0
    return float(top_mean / bot_mean)


# ── Test on one record ────────────────────────────────────────────────────────
sample_hmap = np.load(f"Phase7_Results/heatmaps/{heatmap_files[0]}")
print(f"SCI (Gini):         {spatial_concentration_index(sample_hmap):.4f}")
print(f"Peak/Base ratio:    {peak_variance_ratio(sample_hmap):.4f}")

SCI (Gini):         0.4046
Peak/Base ratio:    3.5678


In [8]:
# Build a lookup: key → heatmap path (fast access)
heatmap_dir   = "Phase7_Results/heatmaps/"
heatmap_lookup = {
    f.replace("_heatmap.npy", ""): os.path.join(heatmap_dir, f)
    for f in os.listdir(heatmap_dir)
}

print(f"Heatmap lookup entries: {len(heatmap_lookup)}")
print(f"Sample lookup key: {list(heatmap_lookup.keys())[0]}")

Heatmap lookup entries: 7000
Sample lookup key: experimental__AM1__s000__cfg0.0


In [9]:
import warnings
warnings.filterwarnings("ignore")

spatial_metrics = []   # will become our analysis dataframe

for rec in tqdm(records, desc="Computing spatial metrics"):
    key  = rec['key']
    hmap = np.load(heatmap_lookup[key])

    sci  = spatial_concentration_index(hmap)
    pvr  = peak_variance_ratio(hmap, top_pct=0.10)

    # Also store: where is the peak variance region (for YOLO overlap later)
    # Peak region = pixels above 90th percentile of this heatmap
    threshold_90 = np.percentile(hmap, 90)
    peak_mask    = (hmap >= threshold_90).astype(np.uint8)  # 64×64 binary mask
    peak_frac    = peak_mask.mean()   # always ~0.10 by construction

    spatial_metrics.append({
        "key"           : key,
        "prompt_id"     : rec['prompt_id'],
        "category"      : rec['category'],
        "cfg"           : rec['cfg'],
        "failure_type"  : rec['failure_type'],
        "hallucinated"  : rec['hallucinated'],
        "global_var"    : rec['variance'],
        "sci_gini"      : sci,
        "peak_var_ratio": pvr,
        "spatial_mean"  : float(hmap.mean()),
        "spatial_max"   : float(hmap.max()),
        "spatial_std"   : float(hmap.std()),
    })

print(f"Done. Total records: {len(spatial_metrics)}")

Computing spatial metrics: 100%|██████████| 7000/7000 [03:25<00:00, 34.08it/s]

Done. Total records: 7000


In [10]:
from scipy import stats

# Split by hallucinated vs clean
hall_sci  = [m['sci_gini']       for m in spatial_metrics if m['hallucinated']]
clean_sci = [m['sci_gini']       for m in spatial_metrics if not m['hallucinated']]
hall_pvr  = [m['peak_var_ratio'] for m in spatial_metrics if m['hallucinated']]
clean_pvr = [m['peak_var_ratio'] for m in spatial_metrics if not m['hallucinated']]

print(f"Hallucinated (n={len(hall_sci)})  — SCI mean: {np.mean(hall_sci):.4f}  std: {np.std(hall_sci):.4f}")
print(f"Clean        (n={len(clean_sci)}) — SCI mean: {np.mean(clean_sci):.4f}  std: {np.std(clean_sci):.4f}")

# Mann-Whitney U test (same as Phase 5 — non-parametric, no normality assumption)
mw_sci = stats.mannwhitneyu(hall_sci, clean_sci, alternative='greater')
mw_pvr = stats.mannwhitneyu(hall_pvr, clean_pvr, alternative='greater')

# Effect size r = Z / sqrt(N)
import math
n_total = len(hall_sci) + len(clean_sci)
z_sci   = stats.norm.ppf(1 - mw_sci.pvalue)
z_pvr   = stats.norm.ppf(1 - mw_pvr.pvalue)
r_sci   = z_sci / math.sqrt(n_total)
r_pvr   = z_pvr / math.sqrt(n_total)

print(f"\n── Spatial Concentration Index (Gini) ──")
print(f"Mann-Whitney U = {mw_sci.statistic:.0f},  p = {mw_sci.pvalue:.3e}")
print(f"Effect size r  = {r_sci:.3f}  {'(Large)' if abs(r_sci)>0.5 else '(Medium)' if abs(r_sci)>0.3 else '(Small)'}")

print(f"\n── Peak Variance Ratio (top 10%) ──")
print(f"Mann-Whitney U = {mw_pvr.statistic:.0f},  p = {mw_pvr.pvalue:.3e}")
print(f"Effect size r  = {r_pvr:.3f}  {'(Large)' if abs(r_pvr)>0.5 else '(Medium)' if abs(r_pvr)>0.3 else '(Small)'}")

Hallucinated (n=574)  — SCI mean: 0.3858  std: 0.0374
Clean        (n=6426) — SCI mean: 0.4358  std: 0.0588

── Spatial Concentration Index (Gini) ──
Mann-Whitney U = 648503,  p = 1.000e+00
Effect size r  = -inf  (Large)

── Peak Variance Ratio (top 10%) ──
Mann-Whitney U = 641923,  p = 1.000e+00
Effect size r  = -inf  (Large)


In [11]:
categories = ["Single-Concept", "Compositional", "Negation", "Ambiguous", "Conflicting"]

print(f"{'Category':<20} {'Hall SCI':>10} {'Clean SCI':>10} {'Diff':>8} {'p-value':>12}")
print("─" * 65)

for cat in categories:
    h = [m['sci_gini'] for m in spatial_metrics
         if m['category'] == cat and m['hallucinated']]
    c = [m['sci_gini'] for m in spatial_metrics
         if m['category'] == cat and not m['hallucinated']]

    if len(h) < 2:
        print(f"{cat:<20} {'N/A':>10} {np.mean(c):>10.4f} {'—':>8} {'N/A':>12}")
        continue

    mw  = stats.mannwhitneyu(h, c, alternative='greater')
    diff = np.mean(h) - np.mean(c)
    print(f"{cat:<20} {np.mean(h):>10.4f} {np.mean(c):>10.4f} {diff:>+8.4f} {mw.pvalue:>12.3e}")

Category               Hall SCI  Clean SCI     Diff      p-value
─────────────────────────────────────────────────────────────────
Single-Concept           0.3829     0.4479  -0.0650    1.000e+00
Compositional            0.3942     0.4637  -0.0695    1.000e+00
Negation                 0.3729     0.4135  -0.0407    1.000e+00
Ambiguous                   N/A     0.4222        —          N/A
Conflicting                 N/A     0.4368        —          N/A


In [12]:
import json

with open("Phase7_Results/spatial_metrics.json", "w") as f:
    json.dump(spatial_metrics, f, indent=2)

print(f"Saved {len(spatial_metrics)} records to Phase7_Results/spatial_metrics.json")

# Quick summary
hall_records = [m for m in spatial_metrics if m['hallucinated']]
clean_records = [m for m in spatial_metrics if not m['hallucinated']]

print(f"\nSummary:")
print(f"  Hallucinated: {len(hall_records)}")
print(f"  Clean:        {len(clean_records)}")
print(f"\n  Hallucinated — mean SCI: {np.mean([m['sci_gini'] for m in hall_records]):.4f}")
print(f"  Clean        — mean SCI: {np.mean([m['sci_gini'] for m in clean_records]):.4f}")
print(f"\n  Hallucinated — mean PVR: {np.mean([m['peak_var_ratio'] for m in hall_records]):.4f}")
print(f"  Clean        — mean PVR: {np.mean([m['peak_var_ratio'] for m in clean_records]):.4f}")

Saved 7000 records to Phase7_Results/spatial_metrics.json

Summary:
  Hallucinated: 574
  Clean:        6426

  Hallucinated — mean SCI: 0.3858
  Clean        — mean SCI: 0.4358

  Hallucinated — mean PVR: 3.3329
  Clean        — mean PVR: 4.1653
